# NB05: External validation strategies (Phase 5R)

LOSO + Cluster-KFold + TargetBinKFold on the opx-liq track with pooled RMSE on
concatenated predictions as the primary metric. Features and hyperparameters
are dynamically loaded from the Phase 3R winning configuration; no feature
set is hard-coded.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)
from src.plot_style import load_winning_config
import warnings
warnings.filterwarnings('ignore')

import ast
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [ ]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr


In [ ]:
import ast
import json
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Dynamic setup: load data and canonical features from Phase 3R winner
df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')

config_3r = load_winning_config(RESULTS)
WIN_FEAT = config_3r['global_feature_set']
print(f'Phase 3R global winning feature set: {WIN_FEAT}')

# Validation uses the UNAUGMENTED feature set (no noise injection during CV)
X, feature_names = build_feature_matrix(df_liq, WIN_FEAT, use_liq=True)
groups_study = df_liq['Citation'].values
clusters = df_liq['chemical_cluster'].values
y_T = df_liq['T_C'].values
y_P = df_liq['P_kbar'].values

print(f'opx-liq rows: {len(df_liq)}, studies: {df_liq["Citation"].nunique()}, clusters: {df_liq["chemical_cluster"].nunique()}, X.shape: {X.shape}')

# Reconstruct best_params_by_model from the seed-42 rows of the canonical
# feature set in the multi-seed results CSV
ms = pd.read_csv(RESULTS / 'nb03_multi_seed_results.csv')
ms_canonical = ms[(ms['split_seed'] == 42) &
                  (ms['feature_set'] == WIN_FEAT) &
                  (ms['track'] == 'opx_liq')]

def _parse_params(s):
    if isinstance(s, dict):
        return s
    try:
        return ast.literal_eval(s)
    except Exception:
        try:
            return json.loads(s)
        except Exception:
            return {}

best_params_by_model = {}
for _, row in ms_canonical.iterrows():
    bp = _parse_params(row['best_params'])
    best_params_by_model.setdefault(row['model_name'], {})[row['target']] = bp

print('Best params reconstructed for models:', sorted(best_params_by_model.keys()))

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor

MODEL_CLASSES = {
    'RF': RandomForestRegressor,
    'ERT': ExtraTreesRegressor,
    'XGB': XGBRegressor,
    'GB': HistGradientBoostingRegressor,
}

def build_model(model_name, params):
    p = dict(params)
    if model_name == 'GB':
        return HistGradientBoostingRegressor(**p, random_state=SEED_MODEL)
    if model_name == 'XGB':
        return XGBRegressor(**p, random_state=SEED_MODEL, n_jobs=-1, verbosity=0)
    return MODEL_CLASSES[model_name](**p, random_state=SEED_MODEL, n_jobs=-1)

## Phase 5.1: Three validation strategies

Runs LOSO, Cluster-KFold, and TargetBinKFold on the canonical opx-liq feature
matrix with the frozen hyperparameters reconstructed above. The primary
metric is pooled RMSE across out-of-fold concatenated predictions; per-fold
RMSE is saved alongside for distributional diagnostics.


In [ ]:
# Phase 5.1: LOSO + Cluster-KFold + TargetBinKFold execution
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

from src.evaluation import (
    loso_splits, cluster_kfold_splits, compute_metrics,
)
from tqdm.auto import tqdm

def pt_grid_labels(df, nT=3, nP=3):
    """Coarse P-T grid for spatial CV. Bin edges are quantile-based so
    fold sizes are roughly balanced."""
    T_bins = pd.qcut(df['T_C'], q=nT, labels=False, duplicates='drop')
    P_bins = pd.qcut(df['P_kbar'], q=nP, labels=False, duplicates='drop')
    return T_bins.astype(str) + '_' + P_bins.astype(str)


STRATEGY_SPLITTERS = {
    'LOSO': ('Citation', loso_splits),
    'Cluster-KFold': ('chemical_cluster', cluster_kfold_splits),
    'TargetBinKFold': ('pt_grid', cluster_kfold_splits),
}

df_liq = df_liq.copy()
df_liq['pt_grid'] = pt_grid_labels(df_liq)

pooled_rows = []
per_fold_rows = []

for strategy, (group_col, split_fn) in tqdm(STRATEGY_SPLITTERS.items(),
                                             total=len(STRATEGY_SPLITTERS),
                                             desc='strategy'):
    groups = df_liq[group_col].values
    splits = split_fn(X, np.zeros(len(X)), groups)
    n_folds = len(splits)
    print(f'\n== {strategy} ({n_folds} folds, group={group_col}) ==')
    for model_name, params_for_model in tqdm(best_params_by_model.items(),
                                                total=len(best_params_by_model),
                                                desc=strategy, leave=False):
        for target_name, y in [('T_C', y_T), ('P_kbar', y_P)]:
            params = params_for_model.get(target_name, {})
            oof = np.full_like(y, np.nan, dtype=float)
            for f_idx, (tr, te) in enumerate(splits):
                est = build_model(model_name, params)
                est.fit(X[tr], y[tr])
                pred = predict_median(est, X[te])
                oof[te] = pred
                fold_rmse = float(np.sqrt(mean_squared_error(y[te], pred)))
                per_fold_rows.append({
                    'strategy': strategy, 'model_name': model_name,
                    'target': target_name, 'fold': f_idx,
                    'n_test': int(len(te)), 'rmse': fold_rmse,
                })
            mask = np.isfinite(oof)
            m = compute_metrics(y[mask], oof[mask])
            pooled_rows.append({
                'strategy': strategy, 'model_name': model_name,
                'target': target_name, 'feature_set': WIN_FEAT,
                'n_folds': n_folds,
                'rmse': m['rmse'], 'mae': m['mae'],
                'r2': m['r2'], 'bias': m['bias'], 'n': m['n'],
            })
            print(f'  {model_name:4s} {target_name:7s}  '
                  f'rmse={m["rmse"]:.2f}  r2={m["r2"]:+.3f}  n={m["n"]}')

pooled_df = pd.DataFrame(pooled_rows)
per_fold_df = pd.DataFrame(per_fold_rows)
pooled_df.to_csv(RESULTS / 'nb05_loso_pooled.csv', index=False)
per_fold_df.to_csv(RESULTS / 'nb05_per_fold_rmse.csv', index=False)
print(f'\nWrote {len(pooled_df)} pooled rows, {len(per_fold_df)} per-fold rows')


In [ ]:
# Phase 5.3: generalization figure (T and P RMSE across three CV strategies)
import matplotlib.pyplot as plt
import json
from config import FAMILY_COLORS
from src.plot_style import apply_style

apply_style()

pooled_df = pd.read_csv(RESULTS / 'nb05_loso_pooled.csv')
per_fold_df = pd.read_csv(RESULTS / 'nb05_per_fold_rmse.csv')

with open(RESULTS / 'nb03_per_family_winners.json') as _f:
    _winners = json.load(_f)
REF_RMSE = {
    ('T_C',    'forest'):  _winners['forest_family']['opx_liq_T_C']['rmse_test_mean'],
    ('T_C',    'boosted'): _winners['boosted_family']['opx_liq_T_C']['rmse_test_mean'],
    ('P_kbar', 'forest'):  _winners['forest_family']['opx_liq_P_kbar']['rmse_test_mean'],
    ('P_kbar', 'boosted'): _winners['boosted_family']['opx_liq_P_kbar']['rmse_test_mean'],
}

STRATS = ['LOSO', 'Cluster-KFold', 'TargetBinKFold']
MODELS = ['RF', 'ERT', 'XGB', 'GB']
FAMILY = {'RF': 'forest', 'ERT': 'forest', 'XGB': 'boosted', 'GB': 'boosted'}
TARGETS = [
    ('T_C',    'T RMSE (°C)'),
    ('P_kbar', 'P RMSE (kbar)'),
]

fold_std = (per_fold_df.groupby(['strategy', 'model_name', 'target'])['rmse']
            .std(ddof=1).reset_index().rename(columns={'rmse': 'rmse_std'}))

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=False)
x = np.arange(len(MODELS))
bar_w = 0.62

for row_idx, (target, ylabel) in enumerate(TARGETS):
    for col_idx, strat in enumerate(STRATS):
        ax = axes[row_idx, col_idx]
        sub = pooled_df[(pooled_df['strategy'] == strat) &
                        (pooled_df['target'] == target)].set_index('model_name')
        std_sub = fold_std[(fold_std['strategy'] == strat) &
                           (fold_std['target'] == target)].set_index('model_name')
        heights = np.array([sub.loc[m, 'rmse'] if m in sub.index else np.nan
                            for m in MODELS], dtype=float)
        errs    = np.array([std_sub.loc[m, 'rmse_std'] if m in std_sub.index else np.nan
                            for m in MODELS], dtype=float)
        colors  = [FAMILY_COLORS[FAMILY[m]] for m in MODELS]

        bars = ax.bar(x, heights, bar_w, yerr=errs, color=colors,
                      edgecolor='black', linewidth=0.6,
                      error_kw={'elinewidth': 1.1, 'capsize': 3, 'ecolor': '#333'})
        for xi, h in zip(x, heights):
            if np.isfinite(h):
                ax.text(xi, h + (0.02 * (np.nanmax(heights) or 1)),
                        f'{h:.1f}' if target == 'T_C' else f'{h:.2f}',
                        ha='center', va='bottom', fontsize=8)

        ax.axhline(REF_RMSE[(target, 'forest')], color=FAMILY_COLORS['forest'],
                   linestyle='--', lw=1.2, alpha=0.85,
                   label=f"NB03 forest test ({REF_RMSE[(target, 'forest')]:.1f})"
                   if target == 'T_C'
                   else f"NB03 forest test ({REF_RMSE[(target, 'forest')]:.2f})")
        ax.axhline(REF_RMSE[(target, 'boosted')], color=FAMILY_COLORS['boosted'],
                   linestyle=':', lw=1.2, alpha=0.85,
                   label=f"NB03 boosted test ({REF_RMSE[(target, 'boosted')]:.1f})"
                   if target == 'T_C'
                   else f"NB03 boosted test ({REF_RMSE[(target, 'boosted')]:.2f})")

        ax.set_xticks(x)
        ax.set_xticklabels(MODELS, fontsize=9)
        ax.set_ylabel(ylabel if col_idx == 0 else '')
        if row_idx == 0:
            ax.set_title(strat, fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(bottom=0)
        ax.legend(loc='upper right', fontsize=7, frameon=True,
                  handlelength=1.6, borderpad=0.4)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, fc=FAMILY_COLORS['forest'], ec='black',
                  label='Forest (RF, ERT)'),
    plt.Rectangle((0, 0), 1, 1, fc=FAMILY_COLORS['boosted'], ec='black',
                  label='Boosted (XGB, GB)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.01), fontsize=10, frameon=False)

fig.suptitle('Generalization under three out-of-distribution CV strategies '
             '(error bars = per-fold std; dashed lines = NB03 random-split '
             'test RMSE)', fontsize=11, fontweight='bold', y=1.00)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])

for _ext in ('png', 'pdf'):
    fig.savefig(FIGURES / f'fig_nb05_generalization.{_ext}',
                dpi=300 if _ext == 'png' else None, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Saved figures/fig_nb05_generalization.{png,pdf} '
      f'({len(STRATS)} strategies x {len(MODELS)} models x {len(TARGETS)} targets).')


In [ ]:
# Phase 5.2: verification
pooled_df = pd.read_csv(RESULTS / 'nb05_loso_pooled.csv')
per_fold_df = pd.read_csv(RESULTS / 'nb05_per_fold_rmse.csv')
assert set(pooled_df['strategy'].unique()) == {'LOSO', 'Cluster-KFold', 'TargetBinKFold'}
assert pooled_df['model_name'].nunique() == 4
assert pooled_df['target'].nunique() == 2
assert (pooled_df['n_folds'] >= 2).all()
assert set(per_fold_df['strategy'].unique()) == {'LOSO', 'Cluster-KFold', 'TargetBinKFold'}

print('=== NB05 complete ===')
print(f'  {len(pooled_df)} pooled rows across 3 strategies x 4 models x 2 targets')
print(f'  {len(per_fold_df)} per-fold rows')
